In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Gold: Fato Alfabetização por Município
# MAGIC
# MAGIC Granularidade: (ano, id_municipio, rede)
# MAGIC Fontes: silver.indicador_municipio + silver.meta_municipio + silver.dim_ibge_municipios
# MAGIC Saída: gold.fato_alfabetizacao_municipio (Delta, partitionBy ano, ZORDER)
# MAGIC Padrão: rastreabilidade + DQ + CTAS (compatível com serverless)

# COMMAND ----------

import sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from pyspark.sql import functions as F

TABELA_META = "silver.meta_municipio"

# COMMAND ----------

# 1. LER AS TABELAS SILVER
df_indicador = spark.table("silver.indicador_municipio")
df_meta      = spark.table(TABELA_META)
df_dim       = spark.table("silver.dim_ibge_municipios")

print(f"[INFO] silver.indicador_municipio : {df_indicador.count():,} linhas")
print(f"[INFO] {TABELA_META} : {df_meta.count():,} linhas")
print(f"[INFO] silver.dim_ibge_municipios: {df_dim.count():,} linhas")

# 2. META DINÂMICA POR ANO (CAST(ano AS INT) para casar com o inteiro do WHEN)
meta_cols = sorted(
    [c for c in df_meta.columns if c.startswith("meta_alfabetizacao_")],
    key=lambda c: int(c.split("_")[-1]),
)

assert meta_cols, f"Nenhuma coluna meta_alfabetizacao_<ano> encontrada em {TABELA_META}"

caso_meta = "CASE " + " ".join(
    f"WHEN CAST(ano AS INT) = {int(c.split('_')[-1])} THEN CAST({c} AS DOUBLE)" for c in meta_cols
) + " ELSE NULL END"

print(f"[INFO] Colunas de meta encontradas: {meta_cols}")

# COMMAND ----------

# 3. JOIN E MONTAGEM DO FATO
cols_dim = set(df_dim.columns)
col_nome = next(
    (c for c in ("nome_municipio", "municipio_nome", "nome") if c in cols_dim), None
)
sel_dim = ["municipio_id", "estado_sigla"] + ([col_nome] if col_nome else [])
df_dim_sel = df_dim.select(*sel_dim)

df_indicador_sel = df_indicador.select(
    "ano", "id_municipio", "rede",
    "taxa_alfabetizacao", "nivel_alfabetizacao", "percentual_participacao",
    "ingested_at", "source", "version",
)
df_meta_sel = df_meta.select(["ano", "id_municipio", "rede"] + meta_cols)

df_fato = (
    df_indicador_sel
    .join(df_meta_sel, ["ano", "id_municipio", "rede"], "left")
    .join(df_dim_sel, F.col("id_municipio") == F.col("municipio_id"), "left")
    .select(
        "ano",
        "id_municipio",
        F.col(col_nome).alias("municipio_nome") if col_nome else F.lit(None).alias("municipio_nome"),
        "estado_sigla",
        "rede",
        F.col("taxa_alfabetizacao").cast("double").alias("resultado"),
        F.expr(caso_meta).alias("meta"),
        "nivel_alfabetizacao",
        "percentual_participacao",
        "ingested_at",
        "source",
        "version",
    )
    .withColumn(
        "status_meta",
        F.when(F.col("meta").isNull(), F.lit("SEM_META"))
         .when(F.col("resultado") >= F.col("meta"), F.lit("ATINGIU"))
         .otherwise(F.lit("NAO_ATINGIU")),
    )
    .withColumn("folga_pp", F.round(F.col("resultado") - F.col("meta"), 2))
    # rede como texto pass-through: se vier descritiva ("Municipal"), mantém;
    # se vier código ("2","3","4"), mapeia
    .withColumn(
        "rede_nome",
        F.when(F.col("rede").isin("2", "3", "4"),
               F.when(F.col("rede") == "2", F.lit("Estadual"))
                .when(F.col("rede") == "3", F.lit("Municipal"))
                .otherwise(F.lit("Privada")))
         .otherwise(F.col("rede")),
    )
)

# COMMAND ----------

# 4. DATA QUALITY
chaves = ["ano", "id_municipio", "rede"]

total = df_fato.count()
nulos_chave = df_fato.filter(
    F.expr(" OR ".join(f"{c} IS NULL" for c in chaves))
).count()
dups = df_fato.groupBy(*chaves).count().filter(F.col("count") > 1).count()
sem_meta = df_fato.filter(F.col("meta").isNull()).count()

print(f"[DQ] Total de linhas: {total}")
print(f"[DQ] Nulos na chave composta: {nulos_chave}")
print(f"[DQ] Duplicados na chave composta: {dups}")
print(f"[DQ] Registros sem meta (SEM_META): {sem_meta}")

spark.sql("CREATE DATABASE IF NOT EXISTS monitoring")
spark.sql("""
  CREATE TABLE IF NOT EXISTS monitoring.dq_results (
    table_name STRING, rule STRING, status STRING,
    records_checked BIGINT, failures BIGINT, run_at TIMESTAMP
  ) USING DELTA
""")

registros = [
    ("gold.fato_alfabetizacao_municipio", "completude_chave",
     "PASS" if nulos_chave == 0 else "FAIL", int(total), int(nulos_chave)),
    ("gold.fato_alfabetizacao_municipio", "unicidade_chave_composta",
     "PASS" if dups == 0 else "FAIL", int(total), int(dups)),
    ("gold.fato_alfabetizacao_municipio", "completude_meta",
     "PASS" if sem_meta == 0 else "FAIL", int(total), int(sem_meta)),
]
df_dq = spark.createDataFrame(
    registros, ["table_name", "rule", "status", "records_checked", "failures"]
).withColumn("run_at", F.current_timestamp())

df_dq.createOrReplaceTempView("vw_dq_gold")
spark.sql("INSERT INTO monitoring.dq_results SELECT * FROM vw_dq_gold")

# COMMAND ----------

# 5. GRAVAR EM DELTA (CTAS, compatível com serverless)
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

df_fato.createOrReplaceTempView("vw_fato_gold")

spark.sql("""
CREATE OR REPLACE TABLE gold.fato_alfabetizacao_municipio
USING DELTA
PARTITIONED BY (ano)
AS SELECT * FROM vw_fato_gold
""")

print(f"\n[OK] gold.fato_alfabetizacao_municipio gravada | Registros: {spark.table('gold.fato_alfabetizacao_municipio').count():,}")

spark.sql("OPTIMIZE gold.fato_alfabetizacao_municipio ZORDER BY (id_municipio, estado_sigla)")
print("[OK] ZORDER aplicado em (id_municipio, estado_sigla)")

# COMMAND ----------

# 6. VERIFICAÇÃO RÁPIDA
spark.sql("""
  SELECT ano,
         COUNT(*)                       AS qtd,
         COUNT(meta)                    AS com_meta,
         SUM(CASE WHEN status_meta = 'ATINGIU' THEN 1 ELSE 0 END) AS atingiram
  FROM gold.fato_alfabetizacao_municipio
  GROUP BY ano
  ORDER BY ano
""").show()